In [27]:
import seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [28]:
transactions = pd.read_csv('transactions_modified.csv')
print(transactions.head())
print(transactions.info())

   step      type      amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0   206  CASH_OUT    62927.08   C473782114           0.00            0.00   
1   380   PAYMENT    32851.57  C1915112886           0.00            0.00   
2   570  CASH_OUT  1131750.38  C1396198422     1131750.38            0.00   
3   184  CASH_OUT    60519.74   C982551468       60519.74            0.00   
4   162   CASH_IN    46716.01  C1759889425     7668050.60      7714766.61   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isPayment  \
0  C2096898696       649420.67       712347.75        0          0   
1   M916879292            0.00            0.00        0          1   
2  C1612235515       313070.53      1444820.92        1          0   
3  C1378644910        54295.32       182654.50        1          0   
4  C2059152908      2125468.75      2078752.75        0          0   

   isMovement  accountDiff  
0           1    649420.67  
1           0         0.00  
2           1    818679.85  


In [29]:
# How many fraudulent transactions?
num_fraud = (transactions['isFraud'] == 1).sum()
print("Number of fraud transactions:", num_fraud)

Number of fraud transactions: 282


In [30]:
type_dict = {"PAYMENT":1, "DEBIT":1}
transactions['isPayment'] = transactions['type'].map(type_dict).fillna(0).astype(int)
movement_dict = {"CASH_OUT":1, "TRANSFER":1}
transactions['isMovement'] = transactions['type'].map(movement_dict).fillna(0).astype(int)
transactions['accountDiff'] = (transactions['oldbalanceOrg'] - transactions['oldbalanceDest']).abs()

In [31]:
features = transactions[['amount', 'isPayment', 'isMovement', 'accountDiff']]
label = transactions['isFraud']

print("Features:\n", features.head())
print("\nLabel:\n", label.head())

Features:
        amount  isPayment  isMovement  accountDiff
0    62927.08          0           1    649420.67
1    32851.57          1           0         0.00
2  1131750.38          0           1    818679.85
3    60519.74          0           1      6224.42
4    46716.01          0           0   5542581.85

Label:
 0    0
1    0
2    1
3    1
4    0
Name: isFraud, dtype: int64


In [32]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.3, random_state=42)

# Normalize the features variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [33]:
# Fit the model to the training data (use scaled features!)
cc_lr = LogisticRegression(max_iter=1000)
cc_lr.fit(X_train_scaled, y_train)

# Score the model on the training data
train_score = cc_lr.score(X_train_scaled, y_train)
print("Training Score (Accuracy):", train_score)

# Score the model on the test data
test_score = cc_lr.score(X_test_scaled, y_test)
print("Testing Score (Accuracy):", test_score)

Training Score (Accuracy): 0.8385714285714285
Testing Score (Accuracy): 0.85


In [34]:
# Print the model coefficients
print("Model coefficients:", cc_lr.coef_)
for feature, coef in zip(features.columns, cc_lr.coef_[0]):
    print(f"{feature}: {coef:.4f}")

# New transaction data
transaction1 = np.array([123456.78, 0.0, 1.0, 54670.1])
transaction2 = np.array([98765.43, 1.0, 0.0, 8524.75])
transaction3 = np.array([543678.31, 1.0, 0.0, 510025.5])

Model coefficients: [[ 2.42110403 -0.61050379  2.10147921 -0.987915  ]]
amount: 2.4211
isPayment: -0.6105
isMovement: 2.1015
accountDiff: -0.9879


In [35]:
# Create a new transaction
your_transaction = np.array([250000.00, 0.0, 1.0, 120000.0])

# Combine new transactions into a single array
sample_transactions = np.array([transaction1, transaction2, transaction3, your_transaction])

# Convert to DataFrame with the same feature names used for training
sample_transactions_df = pd.DataFrame(sample_transactions, columns=features.columns)

# Normalize the new transactions
sample_transactions_scaled = scaler.transform(sample_transactions_df)

In [36]:
# Predict fraud on the new transactions
predictions = cc_lr.predict(sample_transactions_scaled)
probabilities = cc_lr.predict_proba(sample_transactions_scaled)
fraud_probs = probabilities[:, 1]

# Print each prediction with corresponding fraud probability
for i, (pred, prob) in enumerate(zip(predictions, fraud_probs), 1):
    print(f"Sample {i}: Prediction={pred}, Fraud_prob={prob:.4f}")

# Also print arrays for reference
print("Fraud predictions:", predictions)
print("Fraud probabilities (col 1):", fraud_probs)

Sample 1: Prediction=0, Fraud_prob=0.3865
Sample 2: Prediction=0, Fraud_prob=0.0019
Sample 3: Prediction=0, Fraud_prob=0.0035
Sample 4: Prediction=0, Fraud_prob=0.4334
Fraud predictions: [0 0 0 0]
Fraud probabilities (col 1): [0.38647716 0.00193666 0.00354811 0.43336599]
